Часть 2. Марковские цепи. Тест на марковость 0-го vs 1-го порядка

In [ ]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.9 MB/s eta 0:00:00


In [ ]:
import os
import re
import urllib.request
import gzip
import shutil
import numpy as np
from scipy.stats import chi2
from Bio import SeqIO

# chr1.fa (загрузка)
fa_filename = "chr1.fa"
if not os.path.exists(fa_filename):
    url = "http://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr1.fa.gz"
    gz_filename = "chr1.fa.gz"

    urllib.request.urlretrieve(url, gz_filename)

    with gzip.open(gz_filename, 'rb') as f_in:
        with open(fa_filename, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    os.remove(gz_filename)


def get_real_sequence(filename="chr1.fa", length=100000):
    """Загружает первые length нуклеотидов и оставляет только A,C,G,T."""
    record = SeqIO.read(filename, "fasta")
    raw_seq = str(record.seq)[:length].upper()

    clean_seq = re.sub(r'[^ACGT]', '', raw_seq)
    return clean_seq

real_seq = get_real_sequence()
print(f"Длина очищенной реальной последовательности: {len(real_seq)}")

# Случайная последовательность (0-й порядок)
real_counts = {n: real_seq.count(n) for n in 'ACGT'}
N = len(real_seq)
probs = np.array([real_counts[n] for n in 'ACGT'], dtype=float)
probs /= probs.sum()
print("Нормированные частоты нуклеотидов:", dict(zip('ACGT', probs)))

rng = np.random.default_rng(99)
random_seq = ''.join(rng.choice(['A','C','G','T'], size=N, p=probs))

# Функция тестирования
def test_markov_order(seq, label=""):
    nucleotides = ['A','C','G','T']
    idx = {n:i for i,n in enumerate(nucleotides)}
    N = len(seq)
    # Наблюдаемые частоты динуклеотидов O_ij
    O = np.zeros((4,4), dtype=int)
    for i in range(N-1):
        a, b = idx[seq[i]], idx[seq[i+1]]
        O[a,b] += 1

    P_i = np.array([seq.count(n)/N for n in nucleotides])
    E = (N-1) * np.outer(P_i, P_i)
    # Хи-квадрат
    with np.errstate(divide='ignore', invalid='ignore'):
        chi2_stat = np.sum((O - E)**2 / E)
    df = 9
    p_val = 1 - chi2.cdf(chi2_stat, df)
    print(f"\n--- {label} ---")
    print("Наблюдаемые частоты динуклеотидов O:\n", O)
    print("Ожидаемые частоты E:\n", E)
    print(f"Хи-квадрат = {chi2_stat:.3f}, p-value = {p_val:.6f}")
    if p_val < 0.05:
        print(" => Отвергаем гипотезу независимости (нужна модель 1-го порядка)")
    else:
        print(" => Не отвергаем гипотезу независимости (достаточно модели 0-го порядка)")
    return chi2_stat, p_val

chi2_real, p_real = test_markov_order(real_seq, "Реальная последовательность (chr1, очищенная)")
chi2_rand, p_rand = test_markov_order(random_seq, "Случайная последовательность (0-й порядок)")

print("\nИтог:")
print(f"Реальная (chr1): p-value = {p_real:.4f}")
print(f"Случайная: p-value = {p_rand:.4f}")

Длина очищенной реальной последовательности: 90000
Нормированные частоты нуклеотидов: {'A': np.float64(0.2970555555555556), 'C': np.float64(0.22137777777777778), 'G': np.float64(0.20314444444444443), 'T': np.float64(0.27842222222222224)}

--- Реальная последовательность (chr1, очищенная) ---
Наблюдаемые частоты динуклеотидов O:
 [[8989 4786 6253 6707]
 [6879 5461 1101 6482]
 [5208 4143 4749 4183]
 [5659 5534 6180 7685]]
Ожидаемые частоты E:
 [[7941.69203577 5918.46912739 5431.00637704 7443.53540424]
 [5918.46912739 4410.68183632 4047.40493945 5547.22271906]
 [5431.00637704 4047.40493945 3714.04861011 5090.33692896]
 [7443.53540424 5547.22271906 5090.33692896 6976.62652551]]
Хи-квадрат = 4455.112, p-value = 0.000000
 => Отвергаем гипотезу независимости (нужна модель 1-го порядка)

--- Случайная последовательность (0-й порядок) ---
Наблюдаемые частоты динуклеотидов O:
 [[7857 5824 5607 7327]
 [5839 4379 4055 5547]
 [5537 4190 3748 5156]
 [7383 5426 5221 6903]]
Ожидаемые частоты E:
 [[787